In [8]:
import pandas as pd
import numpy as np

df = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital.xlsx")


# 1. Identificar la organización ganadora por distrito y año
ganadores = (
    df.sort_values("total_votos", ascending=False)
      .groupby(["ubigeo", "año"])
      .first()
      .reset_index()
)

ganadores = ganadores[["ubigeo", "año", "organizacion_politica"]]

# 2. Pasar a formato ancho
ganadores_wide = ganadores.pivot(
    index="ubigeo",
    columns="año",
    values="organizacion_politica"
).reset_index()

ganadores_wide.columns = ["ubigeo", "org_2018", "org_2022"]

# 3. Crear turnover (solo conceptualmente)
ganadores_wide["turnover_org"] = (
    ganadores_wide["org_2018"] == ganadores_wide["org_2022"]
).astype(int)

# 4. Unir a la base original
df_final = df.merge(
    ganadores_wide[["ubigeo", "turnover_org"]],
    on="ubigeo",
    how="left"
)

# 5. Dejar turnover solo para 2022
df_final.loc[df_final["año"] == 2018, "turnover_org"] = np.nan

# 6. Guardar la base final en Excel
df_final.to_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_con_turnover.xlsx",
    index=False
)

In [14]:
# 1. Cargar las dos bases
base = pd.read_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_con_turnover.xlsx"
)

den = pd.read_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/denuncias.xlsx"
)

# 2. Suma de delitos por ubigeo
delitos_sum = (
    den.groupby("ubigeo")["cantidad"]
       .sum()
       .reset_index()
       .rename(columns={"cantidad": "suma_delitos"})
)

# 3. Unir por ubigeo
final = base.merge(delitos_sum, on="ubigeo", how="left")

# 4. Guardar base final
final.to_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_final.xlsx",
    index=False
)